In [5]:
#date sep10

import numpy as np
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
#knn model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report

In [6]:
fashion_mnist=fetch_openml('Fashion-MNIST',version=1,as_frame=False)
X=fashion_mnist.data #pixel values
y=fashion_mnist.target.astype(int) #category labels

print(f"Total dataset size: {X.shape[0]} images,each with {X.shape[1]} pixels.")#each image has 28x28 pixels
#our images are greyscale therefore all these 784 pixels contain 0(black)-255(white) values to make a picture

Total dataset size: 70000 images,each with 784 pixels.


In [7]:
#take a balanced subset of 12000 images since 70000 images x 784 pixels can become expensive.
X_subset,_,y_subset,_=train_test_split(
    X,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

In [8]:
X_train,X_test,y_train,y_test=train_test_split(
    X_subset,y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
) #training set=10000,#testing set=2000

print(f"Training images: {X_train.shape[0]}")
print(f"Testing images: {X_test.shape[0]}")

Training images: 10000
Testing images: 2000


In [9]:
print(f"Before scaling -> Min:{X_train.min()},Max:{X_train.max()}")
#rescaling to range [0,1]
X_train=X_train/255.0
X_test=X_test/255.0

print(f"After scaling -> Min:{X_train.min()},Max:{X_train.max()}")

Before scaling -> Min:0,Max:255
After scaling -> Min:0.0,Max:1.0


In [10]:
k_values=[1,3,5,7,9,15]
results={}
print(f"{'K value':<8} | {'Accuracy':<10} | {'Prediction Time (seconds)':<25}")
print ("-"*50)
for k in k_values:
    #initialize the model
    knn=KNeighborsClassifier(n_neighbors=k,metric='euclidean',n_jobs=-1)
    #train the model
    knn.fit(X_train,y_train)
    #predict labels for test images and measure time
    start_time=time.time()
    y_pred=knn.predict(X_test)
    elapsed_time=time.time()-start_time

    acc=accuracy_score(y_test,y_pred)
    results[k]={
        "accuracy":acc,
        "time":elapsed_time,
        "predictions":y_pred
    }
    print(f"{k:<8} | {acc*100:<9.2f}% | {elapsed_time:25.2f}")


K value  | Accuracy   | Prediction Time (seconds)
--------------------------------------------------
1        | 79.30    % |                      2.45
3        | 80.90    % |                      0.20
5        | 81.00    % |                      0.20
7        | 81.10    % |                      0.20
9        | 80.95    % |                      0.20
15       | 80.25    % |                      0.20


In [11]:
# Readable clothing labels mapped to numbers 0-9
class_names=[
    "T-shirts/top","Trousers","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankled boot"
     ]
best_k=max(results,key=lambda k:results[k]["accuracy"])
print(f"Best k is :{best_k} with {results[best_k]['accuracy'] *100:.2f}% accuracy\n")
print("Per-class Classification Report:")
print(classification_report(y_test,results[best_k]["predictions"],target_names=class_names))

Best k is :7 with 81.10% accuracy

Per-class Classification Report:
              precision    recall  f1-score   support

T-shirts/top       0.73      0.83      0.78       200
    Trousers       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
 Ankled boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

